In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
import PIL 
import os
import cv2
import pathlib
import tensorflow_hub as hub
import tf_keras as tfk 

In [ ]:
IMAGE_SHAPE = (128,128)
classifier = tfk.Sequential([
    hub.keras_layer.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-classification/2",
                   input_shape=IMAGE_SHAPE+(3,))
]) 

In [ ]:
dataset_url = "http://download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos',origin=dataset_url,cache_dir='.',untar=True)

In [ ]:
data_dir

In [ ]:
data_dir = pathlib.Path(data_dir)
data_dir

In [ ]:
flowers_images_dict = {
    "roses":list(data_dir.glob("roses/*")),
    "tulips":list(data_dir.glob("tulips/*")),
    "sunflowers":list(data_dir.glob("sunflowers/*")),
    "dandelion":list(data_dir.glob("dandelion/*")),
    "daisy":list(data_dir.glob("daisy/*"))
}

flowers_labels_dict = {
    "roses":0,
    "tulips":1,
    "sunflowers":2,
    "dandelion":3,
    "daisy":4
}

In [ ]:
X,y = [],[]

for flower_name,images in flowers_images_dict.items():
    for image in images:
        img = cv2.imread(str(image))
        resize_img = cv2.resize(img,IMAGE_SHAPE)
        X.append(resize_img)
        y.append(flowers_labels_dict[flower_name])

In [ ]:
X = np.array(X)/255
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
predtrained_model_without_top_layer = tfk.Sequential([
  hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-feature-vector/2",
                 input_shape=IMAGE_SHAPE+(3,), trainable=False)
])

model = tfk.Sequential([
  predtrained_model_without_top_layer,
  tfk.layers.Dense(5,activation='sigmoid')
])
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss = tfk.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
model.fit(X_train,y_train,epochs=5)

In [ ]:
model.evaluate(X_test,y_test)